In [11]:
import torch
from torch.utils import data
from collections import Counter


batch_size = 64
num_steps = 20


data_path = "../data/fra-eng/fra.txt"

In [12]:
def data_read():
    with open(data_path,'r') as file:
        text=file.read()
    return text

In [13]:
def preprocess_nmt(text):
    text = (
        text
        .replace("\u202f", " ")
        .replace("\xa0", " ")
        .lower()
    )
    result=[]
    for i,char in enumerate(text):
        if (
            char in ",.!?"
            and i > 0
            and text[i - 1] != " "
        ):
            result.append(' ')
        result.append(char)
    return "".join(result)

In [14]:
def tokenize_nmt(text,max_text=None):
    source=[]
    target=[]
    for line in text.splitlines():
        parts=line.split('\t')

        if len(parts)<2:
            continue
        source.append(parts[0].split())
        target.append(parts[1].split())
        if(max_text is not None and len(source)>=max_text):
            break
    return source,target

In [15]:
def vocab_maker(tokens,min_freq=2):
    counter=Counter()
    for line in tokens:
        counter.update(line)
    token_freq=sorted(counter.items(),key=lambda x:(-x[1],x[0]))
    idx_to_token=["<unk>",
        "<pad>",
        "<bos>",
        "<eos>"]
    for token,freq in token_freq:
        if freq<min_freq :
            continue
        if token not in idx_to_token:
            idx_to_token.append(token)
    token_to_idx={
        token:idx for idx,token in enumerate(idx_to_token)
    }
    vocab = {
        "idx_to_token": idx_to_token,
        "token_to_idx": token_to_idx
    }

    return vocab

In [16]:
def truncate_pad(line,num_step,pad_idx):
    if len(line)>num_step:
        return line[:num_step]
    return line+[pad_idx]*(num_step-len(line))


In [17]:
def build_array_nmt(lines, vocab, num_steps):
    token_to_idx = vocab["token_to_idx"]

    unk_idx = token_to_idx["<unk>"]
    pad_idx = token_to_idx["<pad>"]
    eos_idx = token_to_idx["<eos>"]

    array = []

    for line in lines:
        line_idx = [
            token_to_idx.get(token, unk_idx)
            for token in line
        ]

        # 每句话末尾添加结束标记
        line_idx.append(eos_idx)

        line_idx = truncate_pad(
            line_idx,
            num_steps,
            pad_idx
        )

        array.append(line_idx)

    array = torch.tensor(
        array,
        dtype=torch.long
    )

    valid_len = (
        array != pad_idx
    ).sum(dim=1)

    return array, valid_len

In [18]:
def load_data_nmt(
    data_path,
    batch_size,
    num_steps,
    num_examples=None
):
    raw_text = data_read()

    text = preprocess_nmt(raw_text)

    source, target = tokenize_nmt(
        text,
        num_examples
    )

    src_vocab = vocab_maker(
        source,
        min_freq=2
    )

    tgt_vocab = vocab_maker(
        target,
        min_freq=2
    )

    src_array, src_valid_len = build_array_nmt(
        source,
        src_vocab,
        num_steps
    )

    tgt_array, tgt_valid_len = build_array_nmt(
        target,
        tgt_vocab,
        num_steps
    )

    dataset = data.TensorDataset(
        src_array,
        src_valid_len,
        tgt_array,
        tgt_valid_len
    )

    train_iter = data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True
    )

    return train_iter, src_vocab, tgt_vocab

In [19]:
train_iter, src_vocab, tgt_vocab = load_data_nmt(
    data_path=data_path,
    batch_size=batch_size,
    num_steps=num_steps,
    num_examples=None
)

In [20]:
for X, X_valid_len, Y, Y_valid_len in train_iter:
    print("X:", X.shape)
    print("X_valid_len:", X_valid_len.shape)
    print("Y:", Y.shape)
    print("Y_valid_len:", Y_valid_len.shape)
    break

X: torch.Size([64, 20])
X_valid_len: torch.Size([64])
Y: torch.Size([64, 20])
Y_valid_len: torch.Size([64])
